# BRISC 2025 · Portable multi-task MRI workflow

Train and evaluate the shared Python package, then launch an independent upload interface. Run setup once after opening a fresh Colab runtime. The repository must include the refactored package (push the local changes first, or upload the project folder).

This is a research demonstration. Perceptual-hash clusters are similarity groups, not verified patient identities. The revised workflow separates model selection from conformal calibration; historical notebook scores are not results of a new run.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab clone the repository; locally, find the checkout from notebooks/.
if Path('/content').is_dir():
    repo = Path('/content/brisc2025-trustworthy-multitask-mri')
    if not repo.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Wilayat-1472/brisc2025-trustworthy-multitask-mri.git', str(repo)], check=True)
else:
    repo = Path.cwd() if (Path.cwd() / 'brisc_mri').is_dir() else Path.cwd().parent
os.chdir(repo)
if not (repo / 'brisc_mri').is_dir():
    raise RuntimeError('Use the refactored checkout containing brisc_mri/. Push local changes to GitHub before cloning in Colab.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data]'], check=True)


## Download and validate the dataset
No absolute Kaggle paths are assumed. The committed pHash-grouped split is retained; its val_cal groups are divided into separate validation and calibration subsets before fitting.

In [ ]:
import kagglehub
from brisc_mri.data import load_manifest, development_splits

download = Path(kagglehub.dataset_download('briscdataset/brisc2025'))
candidates = [download] if (download / 'classification_task').is_dir() else [p.parent for p in download.rglob('classification_task')]
if len(candidates) != 1:
    raise RuntimeError('Set DATA_ROOT manually to the directory containing classification_task and segmentation_task.')
DATA_ROOT = candidates[0]
df = load_manifest('data/brisc2025_clean.csv', DATA_ROOT)
print(dict(zip(('train', 'validation', 'calibration', 'test'), map(len, development_splits(df)))))


## Train and calibrate
Training takes time and benefits from a GPU. Leave RUN_TRAINING=False to use an existing checkpoint. The package saves calibration metadata with new checkpoints. Original raw notebook weights work without conformal sets.

In [ ]:
RUN_TRAINING = False
CHECKPOINT = Path('checkpoints/best_brisc_model.pt')
if RUN_TRAINING:
    subprocess.run([sys.executable, '-m', 'brisc_mri.train', '--data-root', str(DATA_ROOT), '--pretrained', '--output', str(CHECKPOINT)], check=True)
else:
    print('Copy your existing best_brisc_model.pt into', CHECKPOINT.parent)


## Evaluate the frozen checkpoint
Test data is used only for reporting. Calibration comes from the separate held-out subset during training.

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    subprocess.run([sys.executable, '-m', 'brisc_mri.evaluate', '--data-root', str(DATA_ROOT), '--checkpoint', str(CHECKPOINT)], check=True)


## Save weights before ending the Colab runtime
Download the checkpoint using the cell below. Keep model binaries out of ordinary Git commits; attach them to a GitHub Release when sharing. Use `gh auth login` locally for GitHub authentication instead of putting a token in this notebook.

In [ ]:
# Colab only: download the trained checkpoint to your computer.
if Path('/content').is_dir() and CHECKPOINT.is_file():
    from google.colab import files
    files.download(str(CHECKPOINT))
